# Does the gateway-field retention signal replicate? (stress-test demo)

**Context.** This study treats a scientific concept's spread as a network process. An iteration-1 experiment ("exp4") found that
when a concept enters a new field *j*, that field's **gateway centrality** predicts whether the field keeps publishing on the
concept 6-8 years later (field **retention `R`**). Gateway centrality is the eigenvector centrality of *j* on the 26-field
1998-2002 PMI relatedness backbone (`gateway_j`). Adding `gateway_j` raised the leave-one-group-out AUC by **+0.103**. That
estimate came from only 80 episodes and 28 concepts.

**What this artifact does.** The evaluation script `eval.py` makes zero API calls. It stress-tests that lead in pre-registered blocks:

| block | question |
|---|---|
| Step 0 | Does exp4's +0.10254 / +0.10222 reproduce exactly? |
| A | Replication: delta-AUC of `gateway_j` over richer baselines (M0, M1, M2), with a concept-clustered **refit** bootstrap |
| B | Trait confound: does a leave-concept-out field propensity P explain it? Do field intercepts track gateway? |
| C | Placebos: degree-preserving rewiring (C1), node-label permutation (C2), rival centralities with Holm correction (C3) |
| D | O1 artefact re-screen: do the concept-level G-variant gains survive label-coverage controls? |
| E | Power: concept ICC, analytic and simulated minimum detectable effect (MDE) |
| Verdict | Pre-registered ladder: REPLICATES / ATTENUATES / FIELD-TRAIT / FAILS |

**What this demo runs.** The demo loads the **complete exp4 panel** (80 episodes, plus 46 concept rows for Block D) and the 26-field
backbone. It runs the original code of every block on this panel, with smaller resampling counts set in the config cell. The
full run also used the exp1 and exp3 files and their de-duplicated union (362 episodes). Those panels need the iteration-1 raw
files, so the demo shows their results as **reference values** from the full run (`reference_full_run` in the data file). The
full-run verdict is **FAILS**: on the 282 new episodes, the gain over M2 is -0.001 [-0.021, 0.017].

## Setup
Install the dependencies. On Colab, only packages that Colab lacks are installed. Locally, the core packages are pinned to Colab's versions.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT on Colab, always install
_pip('loguru==0.7.3')

# numpy, pandas, scipy, sklearn, networkx, statsmodels, matplotlib — pre-installed on Colab, install locally only
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0',
         'networkx==3.6.1', 'statsmodels==0.14.6')

In [ ]:
# --- original import block of eval.py (argparse / multiprocessing / pickle / resource dropped: no CLI, no process pool,
#     no cache in the notebook) ---
from __future__ import annotations

import json
import math
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from loguru import logger
from scipy.stats import norm, spearmanr

# --- imports of lib.py / harmonise.py / exp4 screen.py ---
import os
import warnings
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
import networkx as nx
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from types import SimpleNamespace

# --- notebook additions: plotting ---
import matplotlib.pyplot as plt

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-8795ea-concepts-spread-once-adopters-make-them/fork/run_AawnV1HvzpNx/round-2/evaluation-1/demo/mini_demo_data.json"
import json
from pathlib import Path

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    local = Path("mini_demo_data.json")
    if local.exists(): return json.loads(local.read_text())
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(data["description"])
print("episodes:", len(data["field_outcomes"]), "| concept rows:", len(data["features"]),
      "| backbone fields:", len(data["field_backbone"]["fields"]))

## Config
All resampling counts are set here. The original production values are in the comments. The demo values keep the whole
notebook under about 10 minutes on a CPU. Draws use the original seeds, so the first *n* draws of each null distribution
match the full run's draws.

In [ ]:
N_BOOT_EXP4 = 40      # refit concept-bootstrap draws for exp4 (Block A)       original: 2000
N_COEF_BOOT = 200     # bootstrap draws of the standardised gateway coefficient  original: 1000
N_WLS_PERM = 1000     # permutations for the stage-2 field-intercept regression  original: 2000
N_REWIRE = 20         # C1 degree-preserving rewired backbones per variant       original: 200
N_PERM = 100          # C2 node-label permutations of the gateway vector         original: 1000
N_BOOT_D = 40         # Block D concept bootstrap draws                          original: 1000
N_SIM = 16            # Block E simulations per (N, m) cell and condition         original: 300
N_PILOT = 16          # Block E calibration pilot simulations per grid point     original: 48
SIM_NS = (1000, 2000, 4000)   # Block E simulated panel sizes                    original: (1000, 2000, 4000)
SIM_MS = (5, 10)              # Block E episodes per concept                     original: (5, 10)

N_WORKERS = 4         # only sets how seeds are chunked (as in eval.py); the notebook runs serially
SEED = 20260928

## exp4's `screen.py` (iteration 1, reused as is)
The evaluation reuses exp4's own statistics, which it imports read-only in the original run. The pieces used here are:
- `logo_predict`: **leave-one-group-out** (LOGO) logistic predictions over the four home groups CS, Eng, BGM and Med. Each
  group is predicted by a model trained on the other three.
- `_prep`: training-fold median imputation.
- `_auc`: AUC.
- `dersimonian_laird`: random-effects pooling.
- `field_level`: the exact routine that produced exp4's +0.10254.

In [ ]:
warnings.filterwarnings("ignore", category=RuntimeWarning)
GROUPS = ["CS", "Eng", "BGM", "Med"]


def _prep(X: pd.DataFrame, train: np.ndarray) -> np.ndarray:
    """Median-impute each column with the TRAINING-fold median (G's missing indicator is a separate column)."""
    X = X.copy()
    for c in X.columns:
        med = X.loc[train, c].median()
        X[c] = X[c].fillna(med if np.isfinite(med) else 0.0)
    return X.values.astype(float)


def logo_predict(df: pd.DataFrame, cols: list[str], y: str, kind: str = "ridge") -> np.ndarray:
    """Leave-one-home-group-out out-of-fold predictions."""
    oof = np.full(len(df), np.nan)
    g = df["group"].values
    for lg in GROUPS:
        te = g == lg
        tr = ~te
        if te.sum() == 0 or tr.sum() < 5:
            continue
        Xall = _prep(df[cols], tr)
        yt = df.loc[tr, y].values
        if kind == "ridge":
            m = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
            m.fit(Xall[tr], yt)
            oof[te] = m.predict(Xall[te])
        else:
            if len(np.unique(yt)) < 2:
                oof[te] = yt.mean()
                continue
            m = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000))
            m.fit(Xall[tr], yt.astype(int))
            oof[te] = m.predict_proba(Xall[te])[:, 1]
    return oof


def _auc(y, p) -> float:
    ok = np.isfinite(p) & np.isfinite(y)
    if ok.sum() < 4 or len(np.unique(y[ok])) < 2:
        return math.nan
    return float(roc_auc_score(y[ok].astype(int), p[ok]))


def dersimonian_laird(est: list[float], var: list[float]) -> dict:
    e = np.array(est, float)
    v = np.array(var, float)
    ok = np.isfinite(e) & np.isfinite(v) & (v > 0)
    e, v = e[ok], v[ok]
    k = len(e)
    if k < 2:
        return {"k": k, "pooled": float(e[0]) if k else math.nan, "se": math.nan, "tau2": math.nan, "I2": math.nan}
    w = 1 / v
    fe = (w * e).sum() / w.sum()
    Q = (w * (e - fe) ** 2).sum()
    C = w.sum() - (w ** 2).sum() / w.sum()
    tau2 = max(0.0, (Q - (k - 1)) / C) if C > 0 else 0.0
    ws = 1 / (v + tau2)
    re = (ws * e).sum() / ws.sum()
    se = math.sqrt(1 / ws.sum())
    I2 = max(0.0, (Q - (k - 1)) / Q) if Q > 0 else 0.0
    return {"k": k, "pooled": float(re), "se": float(se), "tau2": float(tau2), "I2": float(I2), "Q": float(Q)}


def field_level(fr: pd.DataFrame, base: list[str], cand: list[str], n_boot: int = 2000, seed: int = 1) -> dict:
    d = fr.dropna(subset=["R"]).reset_index(drop=True)
    ob = logo_predict(d, base, "R", "logit")
    oc = logo_predict(d, cand, "R", "logit")
    Y = d["R"].values.astype(float)
    ab, ac = _auc(Y, ob), _auc(Y, oc)
    rng = np.random.default_rng(seed)
    cons = d["concept"].unique()
    rows = {c: np.where(d["concept"].values == c)[0] for c in cons}
    boots = []
    for _ in range(n_boot):
        pick = rng.choice(cons, len(cons))
        i = np.concatenate([rows[c] for c in pick])
        a, b = _auc(Y[i], ob[i]), _auc(Y[i], oc[i])
        if np.isfinite(a) and np.isfinite(b):
            boots.append(b - a)
    boots = np.array(boots)
    per = {}
    for g in GROUPS:
        m = d["group"].values == g
        per[g] = {"n_rows": int(m.sum()), "base": _auc(Y[m], ob[m]), "cand": _auc(Y[m], oc[m])}
    return {"n_rows": len(d), "n_concepts": len(cons), "prevalence": float(Y.mean()), "auc_base": ab, "auc_cand": ac,
            "delta_auc": ac - ab, "ci90": [float(np.percentile(boots, 5)), float(np.percentile(boots, 95))],
            "ci95": [float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))], "per_group": per}


# the original imports this file as module `S4`
S4 = SimpleNamespace(GROUPS=GROUPS, _prep=_prep, logo_predict=logo_predict, _auc=_auc,
                     dersimonian_laird=dersimonian_laird, field_level=field_level)

## `lib.py`: the core statistics
- `fast_prep` / `fast_auc`: faster numpy versions of `_prep` and the AUC. They give the same numbers, and Step 0 checks this.
- `propensity`: the **leave-concept-out field retention propensity P**. It is a shrunken mean of R for the field, computed
  inside each training fold so that it does not leak. It is the "field trait" rival used in Block B1.
- `eval_specs`: fits every (baseline, baseline + candidate) pair with LOGO. It returns the pooled and per-group delta-AUC.
- `resample` / `boot_worker`: the **concept-clustered refit bootstrap**. Every draw resamples concepts and refits all models.
- `perm_worker`: recomputes delta-AUC for placebo gateway vectors (Blocks C1 and C2).
- `o1_base`, `logo_concept`, `concept_specs_eval`, `concept_boot_worker`: the concept-level O1 models for Block D.
- `sim_worker`: simulates clustered episodes for the Block E power analysis.

In [ ]:
GROUPS = S4.GROUPS  # ["CS", "Eng", "BGM", "Med"]
_ORIG_PREP, _ORIG_AUC = S4._prep, S4._auc


def fast_prep(X: pd.DataFrame, train: np.ndarray) -> np.ndarray:
    """Numerically identical numpy version of screen._prep (training-fold median imputation per column)."""
    A = X.to_numpy(dtype=float, copy=True)
    if np.isnan(A).any():
        with np.errstate(all="ignore"):
            import warnings
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                med = np.nanmedian(A[train], axis=0)
        med = np.where(np.isfinite(med), med, 0.0)
        r, c = np.nonzero(np.isnan(A))
        A[r, c] = med[c]
    return A


def fast_auc(y, p) -> float:
    """Rank (Mann-Whitney) AUC with screen._auc's conventions (finite rows, >= 4 rows, both classes)."""
    from scipy.stats import rankdata
    y = np.asarray(y, float)
    p = np.asarray(p, float)
    ok = np.isfinite(p) & np.isfinite(y)
    if ok.sum() < 4:
        return math.nan
    y, p = y[ok], p[ok]
    n1 = y.sum()
    n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return math.nan
    r = rankdata(p)
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


# Speed: exp4's logo_predict looks up _prep at call time; the numpy version gives identical matrices (verified in
# eval.py against the original on every dataset before use).
S4._prep = fast_prep
_prep = fast_prep  # notebook: logo_predict above resolves the global name `_prep`, so patch it here too


# ----------------------------------------------------------------------------- fold-dependent features
def propensity(key: np.ndarray, cl: np.ndarray, y: np.ndarray, grp: np.ndarray, test_g: str, a: float,
               pool: pd.DataFrame | None = None, concept: np.ndarray | None = None) -> np.ndarray:
    """Shrunken leave-concept-out field retention propensity for one LOGO fold.

    Within-dataset (pool None): statistics from training-fold rows (group != test_g) of the same field key;
    test rows use all training rows of the key, training rows exclude rows of their own cluster (concept copy).
    Pooled: statistics from the external pool (rows of all three files) with group != test_g and concept != own
    concept, for both test and training rows. P = (sum R + a*pbar) / (n + a); pbar = training mean; n=0,a=0 -> pbar.
    """
    n = len(y)
    out = np.empty(n)
    if pool is None:
        tr = grp != test_g
        pbar = float(y[tr].mean())
        d = pd.DataFrame({"k": key[tr], "c": cl[tr], "y": y[tr]})
        tot = d.groupby("k")["y"].agg(["sum", "count"])
        byc = d.groupby(["k", "c"])["y"].agg(["sum", "count"])
        ks = tot.reindex(key)
        s_all = np.nan_to_num(ks["sum"].to_numpy(float))
        n_all = np.nan_to_num(ks["count"].to_numpy(float))
        own = byc.reindex(pd.MultiIndex.from_arrays([key, cl]))
        s_own = np.nan_to_num(own["sum"].to_numpy(float))
        n_own = np.nan_to_num(own["count"].to_numpy(float))
        s = np.where(tr, s_all - s_own, s_all)
        c = np.where(tr, n_all - n_own, n_all)
    else:
        pp = pool[pool["group"].to_numpy() != test_g]
        pbar = float(pp["R"].mean())
        tot = pp.groupby("key")["R"].agg(["sum", "count"])
        byc = pp.groupby(["key", "concept"])["R"].agg(["sum", "count"])
        ks = tot.reindex(key)
        own = byc.reindex(pd.MultiIndex.from_arrays([key, concept]))
        s = np.nan_to_num(ks["sum"].to_numpy(float)) - np.nan_to_num(own["sum"].to_numpy(float))
        c = np.nan_to_num(ks["count"].to_numpy(float)) - np.nan_to_num(own["count"].to_numpy(float))
    den = c + a
    out[:] = np.where(den > 0, (s + a * pbar) / np.where(den > 0, den, 1.0), pbar)
    return out


def logo_ext(df: pd.DataFrame, cols: list[str], y: str = "R", a: float = 2.0, pool: pd.DataFrame | None = None,
             cl_col: str = "cl") -> np.ndarray:
    """exp4 screen.logo_predict(kind='logit') plus fold-computed P columns ('P_within', 'P_pooled')."""
    dyn = [c for c in cols if c in ("P_within", "P_pooled")]
    if not dyn:
        return S4.logo_predict(df, cols, y, "logit")
    oof = np.full(len(df), np.nan)
    g = df["group"].to_numpy()
    Y = df[y].to_numpy(float)
    key = df["key"].to_numpy()
    cl = df[cl_col].to_numpy()
    con = df["concept"].to_numpy()
    for lg in GROUPS:
        te = g == lg
        tr = ~te
        if te.sum() == 0 or tr.sum() < 5:
            continue
        X = df[[c for c in cols if c not in dyn]].copy()
        if "P_within" in dyn:
            X["P_within"] = propensity(key, cl, Y, g, lg, a)
        if "P_pooled" in dyn:
            X["P_pooled"] = propensity(key, cl, Y, g, lg, a, pool=pool, concept=con)
        X = X[cols]
        Xall = S4._prep(X, tr)
        yt = Y[tr]
        if len(np.unique(yt)) < 2:
            oof[te] = yt.mean()
            continue
        m = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000))
        m.fit(Xall[tr], yt.astype(int))
        oof[te] = m.predict_proba(Xall[te])[:, 1]
    return oof


def pooled_auc(y: np.ndarray, p: np.ndarray, g: np.ndarray) -> tuple[float, int]:
    """Pooled OOF AUC after dropping rows of test groups that contain a single class. Returns (auc, n_dropped_groups)."""
    keep = np.ones(len(y), bool)
    nd = 0
    for lg in GROUPS:
        m = g == lg
        if m.sum() and len(np.unique(y[m])) < 2:
            keep &= ~m
            nd += 1
    return fast_auc(y[keep], p[keep]), nd


def group_aucs(y: np.ndarray, p: np.ndarray, g: np.ndarray) -> dict:
    return {lg: fast_auc(y[g == lg], p[g == lg]) for lg in GROUPS}


def eval_specs(df: pd.DataFrame, specs: list[tuple[str, list[str], list[str]]], a: float = 2.0,
               pool: pd.DataFrame | None = None, keep_oof: bool = False) -> dict:
    """Fit every distinct model once; return pooled/per-group delta-AUC per spec."""
    cache: dict[tuple, np.ndarray] = {}
    Y = df["R"].to_numpy(float)
    g = df["group"].to_numpy()

    def get(cols):
        k = tuple(cols)
        if k not in cache:
            cache[k] = logo_ext(df, list(cols), "R", a, pool)
        return cache[k]
    out = {}
    for name, base, cand in specs:
        ob, oc = get(base), get(cand)
        ab, nd = pooled_auc(Y, ob, g)
        ac, _ = pooled_auc(Y, oc, g)
        gb, gc = group_aucs(Y, ob, g), group_aucs(Y, oc, g)
        r = {"auc_base": ab, "auc_cand": ac, "delta": ac - ab, "n_groups_dropped": nd,
             "per_group": {lg: {"base": gb[lg], "cand": gc[lg],
                                "delta": (gc[lg] - gb[lg]) if np.isfinite(gb[lg]) and np.isfinite(gc[lg]) else math.nan}
                           for lg in GROUPS},
             "brier_base": float(np.nanmean((ob - Y) ** 2)), "brier_cand": float(np.nanmean((oc - Y) ** 2))}
        if keep_oof:
            r["oof_base"], r["oof_cand"] = ob, oc
        out[name] = r
    return out


def resample(df: pd.DataFrame, rng: np.random.Generator, stratified: bool = False) -> pd.DataFrame:
    """Concept-clustered resample; every duplicated concept gets a fresh cluster id ('cl')."""
    cons = df["concept"].unique()
    rows = {c: np.flatnonzero(df["concept"].to_numpy() == c) for c in cons}
    if stratified:
        cg = df.groupby("concept")["group"].first()
        pick = np.concatenate([rng.choice(cg.index[cg == lg].to_numpy(), (cg == lg).sum())
                               for lg in GROUPS if (cg == lg).sum()])
    else:
        pick = rng.choice(cons, len(cons))
    idx, cid = [], []
    for k, c in enumerate(pick):
        idx.append(rows[c])
        cid.append(np.full(len(rows[c]), k))
    d = df.iloc[np.concatenate(idx)].reset_index(drop=True)
    d["cl"] = np.concatenate(cid)
    return d


def boot_worker(args: tuple) -> list[dict]:
    """One chunk of refit bootstrap draws. args = (df, specs, seeds, a, pool, stratified)."""
    df, specs, seeds, a, pool, stratified = args
    res = []
    for sd in seeds:
        rng = np.random.default_rng(sd)
        d = resample(df, rng, stratified)
        r = eval_specs(d, specs, a, pool)
        res.append({k: {"delta": v["delta"], "nd": v["n_groups_dropped"],
                        "pg": {lg: v["per_group"][lg]["delta"] for lg in GROUPS}} for k, v in r.items()})
    return res


def perm_worker(args: tuple) -> list[dict]:
    """Placebo gateway vectors: args = (df, W, base_cols, gvecs, a). Returns point delta-AUC per vector for each base."""
    df, W, bases, gvecs = args
    Y = df["R"].to_numpy(float)
    g = df["group"].to_numpy()
    base_auc = {nm: pooled_auc(Y, S4.logo_predict(df, cols, "R", "logit"), g)[0] for nm, cols in bases.items()}
    out = []
    for gv in gvecs:
        d = df.copy()
        d["g_plac"] = W @ gv
        r = {}
        for nm, cols in bases.items():
            r[nm] = pooled_auc(Y, S4.logo_predict(d, cols + ["g_plac"], "R", "logit"), g)[0] - base_auc[nm]
        out.append(r)
    return out


# ----------------------------------------------------------------------------- concept-level O1 (Block D)
def o1_base(home: np.ndarray, cl: np.ndarray, y: np.ndarray, grp: np.ndarray, test_g: str) -> np.ndarray:
    """Training-fold mean O1 of concepts sharing the home field (leave-own-concept-out for training rows),
    falling back to the training-fold global mean."""
    tr = grp != test_g
    out = np.empty(len(y))
    gm = y[tr].mean()
    for i in range(len(y)):
        m = tr & (home == home[i]) & (cl != cl[i])
        out[i] = y[m].mean() if m.sum() else (y[tr & (cl != cl[i])].mean() if tr[i] else gm)
    return out


def logo_concept(df: pd.DataFrame, cols: list[str], y: str) -> np.ndarray:
    dyn = "O1_base" in cols
    if not dyn:
        return S4.logo_predict(df, cols, y, "logit")
    oof = np.full(len(df), np.nan)
    g = df["group"].to_numpy()
    Y = df[y].to_numpy(float)
    for lg in GROUPS:
        te = g == lg
        tr = ~te
        if te.sum() == 0 or tr.sum() < 5:
            continue
        X = df[[c for c in cols if c != "O1_base"]].copy()
        X["O1_base"] = o1_base(df["home"].to_numpy(), df["cl"].to_numpy(), Y, g, lg)
        X = X[cols]
        Xa = S4._prep(X, tr)
        yt = Y[tr]
        if len(np.unique(yt)) < 2:
            oof[te] = yt.mean()
            continue
        m = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000))
        m.fit(Xa[tr], yt.astype(int))
        oof[te] = m.predict_proba(Xa[te])[:, 1]
    return oof


def concept_specs_eval(df: pd.DataFrame, specs: list[tuple[str, list[str], list[str]]], y: str = "O1") -> dict:
    cache = {}
    Y = df[y].to_numpy(float)
    g = df["group"].to_numpy()

    def get(cols):
        k = tuple(cols)
        if k not in cache:
            cache[k] = logo_concept(df, list(cols), y)
        return cache[k]
    out = {}
    for nm, b, c in specs:
        ob, oc = get(b), get(c)
        ab, ac = fast_auc(Y, ob), fast_auc(Y, oc)
        pg = {}
        for lg in GROUPS:
            m = g == lg
            x1, x2 = fast_auc(Y[m], ob[m]), fast_auc(Y[m], oc[m])
            pg[lg] = x2 - x1 if np.isfinite(x1) and np.isfinite(x2) else math.nan
        out[nm] = {"base": ab, "cand": ac, "delta": ac - ab, "per_group": pg}
    return out


def concept_boot_worker(args: tuple) -> list[dict]:
    df, specs, seeds = args
    res = []
    for sd in seeds:
        rng = np.random.default_rng(sd)
        idx = rng.integers(0, len(df), len(df))
        d = df.iloc[idx].reset_index(drop=True)
        d["cl"] = np.arange(len(d))
        r = concept_specs_eval(d, specs)
        res.append({k: v["delta"] for k, v in r.items()})
    return res


# ----------------------------------------------------------------------------- Block E simulation
def sim_worker(args: tuple) -> list[float]:
    """Simulate clustered episodes from the fitted M2+gateway model and return sampling draws of 4-fold grouped-CV
    delta-AUC. args = (Xpool, beta0, beta, gcol, sigma_c, N, m, seeds)."""
    from sklearn.model_selection import GroupKFold
    Xpool, b0, beta, gcol, sig, N, m, seeds = args[:8]
    keyc, tau_f = (args[8], args[9]) if len(args) > 8 else (None, 0.0)
    out = []
    for sd in seeds:
        rng = np.random.default_rng(sd)
        nc = int(math.ceil(N / m))
        idx = rng.integers(0, len(Xpool), nc * m)
        X = Xpool[idx]
        conc = np.repeat(np.arange(nc), m)
        u = rng.normal(0, sig, nc)[conc]
        eta = b0 + X @ beta + u
        if keyc is not None and tau_f > 0:  # field random intercept: arbitrary field constants can absorb it
            eta = eta + rng.normal(0, tau_f, int(keyc.max()) + 1)[keyc[idx]]
        y = rng.random(len(eta)) < 1 / (1 + np.exp(-eta))
        if y.all() or (~y).all():
            continue
        pb = np.full(len(y), np.nan)
        pc = np.full(len(y), np.nan)
        base_cols = [i for i in range(X.shape[1]) if i != gcol]
        for tr, te in GroupKFold(4).split(X, y, conc):
            if len(np.unique(y[tr])) < 2:
                continue
            for cols, dest in ((base_cols, pb), (list(range(X.shape[1])), pc)):
                mdl = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000))
                mdl.fit(X[tr][:, cols], y[tr])
                dest[te] = mdl.predict_proba(X[te][:, cols])[:, 1]
        out.append(fast_auc(y.astype(float), pc) - fast_auc(y.astype(float), pb))
    return out


# the original imports this file as module `lib`
lib = SimpleNamespace(**{k: globals()[k] for k in (
    "GROUPS", "_ORIG_PREP", "_ORIG_AUC", "fast_prep", "fast_auc", "propensity", "logo_ext", "pooled_auc", "group_aucs",
    "eval_specs", "resample", "boot_worker", "perm_worker", "o1_base", "logo_concept", "concept_specs_eval",
    "concept_boot_worker", "sim_worker")})

## `harmonise.py`: the 26-field backbone and the exp4 panel (Step 0)
`Backbone` holds the 1998-2002 field-relatedness matrix `phi` (positive PMI between the 26 OpenAlex fields), the gateway
vector (weighted eigenvector centrality) and the **rival centralities** used in Block C3: strength, degree, betweenness,
PageRank, closeness, k-core, the eigenvector on `phi_min`, and field size. `_attach` projects each field-level quantity onto
the episodes through a weight row `W`. `load_exp4` builds the harmonised exp4 panel. The feature families are:
- **M0**: the field's own early baseline (log count, growth, share).
- **B5**: five concept-level controls (volume, growth, off-home share, entropy, reach).
- **REL**: log field size, relatedness to the concept's home field (`phi_home_j`) and relatedness density.

*Notebook change:* the backbone JSON and the two exp4 CSVs are read from the loaded `data` dict, not from the iteration-1 folder.

In [ ]:
M0 = ["b_logn", "b_growth", "b_share"]
B5 = ["b5_logvol", "b5_growth", "b5_offhome", "b5_entropy", "b5_reach"]
REL = ["log_field_size", "phi_home_j", "density_j"]
RIVALS = ["r_strength", "r_degree", "r_betweenness", "r_pagerank", "r_closeness", "r_kcore", "r_eig_phimin",
          "log_field_size"]


class Backbone:
    def __init__(self) -> None:
        b = data["field_backbone"]  # notebook: was json.loads((E4 / "field_backbone.json").read_text())
        self.raw = b
        self.fields = b["fields"]
        self.idx = {f: i for i, f in enumerate(self.fields)}
        self.phi = np.array(b["phi"])
        self.phi_min = np.array(b["phi_min"])
        self.n = np.array(b["n_field"], float)
        self.logn = np.log(self.n)
        self.gate = np.array(b["gateway_eig"])
        self.domain = b["domain"]
        self.rivals = self._rivals()

    def graph(self) -> nx.Graph:
        G = nx.Graph()
        G.add_nodes_from(range(26))
        for i in range(26):
            for j in range(i + 1, 26):
                if self.phi[i, j] > 0:
                    G.add_edge(i, j, weight=self.phi[i, j], dist=1.0 / self.phi[i, j])
        return G

    def _rivals(self) -> dict[str, np.ndarray]:
        G = self.graph()
        v = lambda d: np.array([d[i] for i in range(26)], float)  # noqa: E731
        return {"gateway_j": self.gate,
                "r_strength": v(dict(G.degree(weight="weight"))),
                "r_degree": v(dict(G.degree())),
                "r_betweenness": v(nx.betweenness_centrality(G, weight="dist")),
                "r_pagerank": v(nx.pagerank(G, alpha=0.85, weight="weight")),
                "r_closeness": v(nx.closeness_centrality(G, distance="dist")),
                "r_kcore": v(nx.core_number(G)),
                "r_eig_phimin": np.array(self.raw["gateway_eig_phimin"]),
                "log_field_size": self.logn}


def _w_row(bb: Backbone, names: list[str]) -> np.ndarray:
    w = np.zeros(26)
    ii = [bb.idx[n] for n in names]
    w[ii] = bb.n[ii] / bb.n[ii].sum()
    return w


def _attach(df: pd.DataFrame, W: np.ndarray, homes: list[list[int]], bb: Backbone, K: list[set[int]] | None) -> None:
    """gateway, rivals, log_field_size, phi_home_j and (if K given) density_j from weight rows W."""
    for nm, vec in bb.rivals.items():
        df[nm] = W @ vec
    df["phi_home_j"] = [float(np.mean([bb.phi[h] @ w for h in hs])) if hs else 0.0 for w, hs in zip(W, homes)]
    if K is not None:
        dens = []
        colsum = bb.phi.sum(0)
        for w, Kc in zip(W, K):
            val = 0.0
            for k in np.flatnonzero(w):
                Kj = list(Kc - {k})
                val += w[k] * (bb.phi[Kj, k].sum() / colsum[k] if Kj and colsum[k] > 0 else 0.0)
            dens.append(val)
        df["density_j"] = dens


def _K_from_rows(df: pd.DataFrame, W: np.ndarray, homes: list[list[int]]) -> list[set[int]]:
    """Early-window field presence approximated by the concept's home field(s) plus every field with a retention row
    (rows require >= 5 early papers; exp4's own rule is >= 2 labelled papers, which the exp1/exp3 files do not hold)."""
    pres: dict[str, set[int]] = {}
    for c, w, hs in zip(df["concept"], W, homes):
        pres.setdefault(c, set(hs)).update(np.flatnonzero(w).tolist())
    return [pres[c] for c in df["concept"]]


def load_exp4(bb: Backbone) -> tuple[pd.DataFrame, np.ndarray, dict]:
    fr = pd.DataFrame(data["field_outcomes"])  # notebook: was pd.read_csv(E4 / "field_outcomes.csv")
    ft = pd.DataFrame(data["features"])        # notebook: was pd.read_csv(E4 / "features.csv")
    ft = ft[["concept", "t0", "home", "log_count_W5", "growth_W5_B5", "offhome_share_W3", "entropy_W3", "reach_W3"]]
    d = fr.merge(ft, on="concept", how="left", validate="many_to_one")
    W = np.vstack([_w_row(bb, [f]) for f in d["field"]])
    homes = [[bb.idx[h] for h in str(hs).split(";") if h in bb.idx] for hs in d["home"]]
    orig = d[["gateway_j", "phi_home_j", "log_field_size", "density_j"]].copy()
    out = pd.DataFrame({"concept": d["concept"], "group": d["group"], "key": d["field"], "dkey": d["field"],
                        "source": "exp4", "R": d["R"].astype(float), "t0": d["t0"].astype(float),
                        "b_logn": d["log_n_W3"], "b_growth": d["growth_j"], "b_share": d["share_W3"],
                        "b5_logvol": d["log_count_W5"], "b5_growth": d["growth_W5_B5"],
                        "b5_offhome": d["offhome_share_W3"], "b5_entropy": d["entropy_W3"], "b5_reach": d["reach_W3"],
                        "n_early": d["n_W3"].astype(float), "one_to_many": 0, "many_to_one": 0,
                        "field_s2": ""})
    _attach(out, W, homes, bb, None)
    Kapprox = _K_from_rows(out, W, homes)
    tmp = out.copy()
    _attach(tmp, W, homes, bb, Kapprox)
    chk = {"gateway_j_maxabs": float(np.abs(out["gateway_j"] - orig["gateway_j"]).max()),
           "phi_home_j_maxabs": float(np.abs(out["phi_home_j"] - orig["phi_home_j"]).max()),
           "log_field_size_maxabs": float(np.abs(out["log_field_size"] - orig["log_field_size"]).max()),
           "density_j_rows_approx_vs_exp4": {"spearman": float(pd.Series(tmp["density_j"]).corr(orig["density_j"],
                                                                                                method="spearman")),
                                             "maxabs": float(np.abs(tmp["density_j"] - orig["density_j"]).max())}}
    out["density_j"] = orig["density_j"].to_numpy()  # exp4's own K (>=2 labelled papers in t0..t0+2) is authoritative
    return out, W, chk


# the original imports this file as module `H`
H = SimpleNamespace(M0=M0, B5=B5, REL=REL, RIVALS=RIVALS, Backbone=Backbone, load_exp4=load_exp4)

## `eval.py` helpers
These are the original helper functions:
- `ci`: percentile confidence intervals.
- `holm`: Holm multiple-testing correction.
- `spec_set`: the model ladder. Each entry is (baseline, baseline + candidate), for example M2 against M2 + `gateway_j`.
- `run_boot` / `summarise_boot`: the refit bootstrap and its summary.
- `std_coef_boot`: the standardised gateway coefficient with a bootstrap CI.
- `field_effects` / `wls_perm` / `mixed_icc`: Block B2.
- `_connected` / `rewire` / `eig_from_edges`: the C1 rewiring null.

*Notebook change:* `pool_map` runs serially with `map`, because a spawn process pool cannot pickle functions defined in a
notebook. The draws and results are unchanged. The on-disk `cached(...)` wrapper is dropped, so each job is called directly.

In [ ]:
def pool_map(fn, jobs: list) -> list:
    """Notebook version: serial map (the original used one shared spawn-context ProcessPoolExecutor)."""
    return list(map(fn, jobs))


def chunks(seeds: list[int], k: int) -> list[list[int]]:
    return [list(x) for x in np.array_split(np.array(seeds), k) if len(x)]


def ci(x: np.ndarray) -> dict:
    x = np.asarray([v for v in x if np.isfinite(v)])
    if not len(x):
        return {"n": 0}
    return {"n": int(len(x)), "sd": float(x.std(ddof=1)), "ci90": [float(np.percentile(x, 5)), float(np.percentile(x, 95))],
            "ci95": [float(np.percentile(x, 2.5)), float(np.percentile(x, 97.5))],
            "p_le0": float(np.mean(x <= 0)), "p_two_sided": float(min(1.0, 2 * min(np.mean(x <= 0), np.mean(x >= 0))))}


def holm(ps: dict[str, float]) -> dict[str, float]:
    items = sorted(ps.items(), key=lambda kv: kv[1])
    m = len(items)
    out, run = {}, 0.0
    for i, (k, p) in enumerate(items):
        run = max(run, min(1.0, (m - i) * p))
        out[k] = run
    return out


# ----------------------------------------------------------------------------- spec sets
def spec_set(ds: str, rivals: bool) -> tuple[list, dict]:
    M0 = H.M0 + (["src_exp1", "src_exp3"] if ds in ("union", "union_agree", "new_eps") else [])
    M1 = M0 + H.B5
    M2 = M1 + H.REL + (["bg_LOR_j"] if ds.startswith("exp1") else [])
    g = "gateway_j"
    sp = [("M0", M0, M0 + [g]), ("M1", M1, M1 + [g]), ("M2", M2, M2 + [g]),
          ("M2+P", M2 + ["P_within"], M2 + ["P_within", g]), ("P_alone", M2, M2 + ["P_within"]),
          ("M2+Ppool", M2 + ["P_pooled"], M2 + ["P_pooled", g]), ("Ppool_alone", M2, M2 + ["P_pooled"])]
    if rivals:
        sp += [(f"rival:{r}", M2, M2 + [r]) for r in H.RIVALS if r != "log_field_size"]
        # log_field_size is already inside M2 -> its rival test is M2 without it vs M2
        m2ns = [c for c in M2 if c != "log_field_size"]
        sp += [("rival:log_field_size", m2ns, M2), ("gateway_vs_M2_minus_size", m2ns, m2ns + [g])]
    return sp, {"M0": M0, "M1": M1, "M2": M2}


def run_boot(df: pd.DataFrame, specs: list, n: int, pool: pd.DataFrame, seed: int, stratified: bool = False) -> list:
    seeds = list(range(seed, seed + n))
    jobs = [(df, specs, c, 2.0, pool, stratified) for c in chunks(seeds, N_WORKERS * 4)]
    out = []
    for r in pool_map(lib.boot_worker, jobs):
        out.extend(r)
    return out


def summarise_boot(point: dict, boots: list, specs: list) -> dict:
    res = {}
    for nm, _, _ in specs:
        arr = np.array([b[nm]["delta"] for b in boots], float)
        nd = int(sum(b[nm]["nd"] > 0 for b in boots))
        pg = {lg: ci(np.array([b[nm]["pg"][lg] for b in boots], float)) for lg in GROUPS}
        p = point[nm]
        res[nm] = {"auc_base": p["auc_base"], "auc_cand": p["auc_cand"], "delta": p["delta"],
                   "brier_change": p["brier_cand"] - p["brier_base"],
                   "refit_boot": {**ci(arr), "n_draws": len(boots), "draws_with_single_class_test_group": nd,
                                  "share_single_class_draws": nd / max(1, len(boots))},
                   "per_group": {lg: {**p["per_group"][lg], "boot": pg[lg]} for lg in GROUPS},
                   "n_groups_positive": int(sum(1 for lg in GROUPS if np.isfinite(p["per_group"][lg]["delta"])
                                                and p["per_group"][lg]["delta"] > 0)),
                   "n_groups_evaluable": int(sum(1 for lg in GROUPS if np.isfinite(p["per_group"][lg]["delta"])))}
    return res


def std_coef_boot(df: pd.DataFrame, cols: list[str], target: str, n: int, seed: int) -> dict:
    """Standardised logistic coefficient of `target` in a full-sample fit, concept-clustered bootstrap CI."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler

    def fit(d):
        X = S4._prep(d[cols], np.ones(len(d), bool))
        X = StandardScaler().fit_transform(X)
        y = d["R"].to_numpy(int)
        if len(np.unique(y)) < 2:
            return math.nan
        return float(LogisticRegression(C=1.0, max_iter=1000).fit(X, y).coef_[0][cols.index(target)])
    b0 = fit(df)
    rng = np.random.default_rng(seed)
    bs = [fit(lib.resample(df, rng)) for _ in range(n)]
    return {"coef_std": b0, **ci(np.array(bs))}


# ----------------------------------------------------------------------------- Block B2 helpers
def field_effects(df: pd.DataFrame, M1: list[str]) -> pd.DataFrame:
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    X = StandardScaler().fit_transform(S4._prep(df[M1], np.ones(len(df), bool)))
    keys = sorted(df["key"].unique())
    D = (df["key"].to_numpy()[:, None] == np.array(keys)[None, :]).astype(float)
    m = LogisticRegression(C=1.0, max_iter=2000).fit(np.hstack([X, D]), df["R"].to_numpy(int))
    u = m.coef_[0][X.shape[1]:]
    cnt = df["key"].value_counts()
    return pd.DataFrame({"key": keys, "u": u, "n": [int(cnt[k]) for k in keys],
                         "gateway_j": [float(df.loc[df["key"] == k, "gateway_j"].iloc[0]) for k in keys],
                         "log_field_size": [float(df.loc[df["key"] == k, "log_field_size"].iloc[0]) for k in keys]})


def wls_perm(fe: pd.DataFrame, xcol: str, n_perm: int, seed: int) -> dict:
    import statsmodels.api as sm
    f = fe[fe["n"] >= 3].reset_index(drop=True)
    if len(f) < 4:
        return {"n_fields": int(len(f)), "note": "fewer than 4 fields with >= 3 rows"}
    X = sm.add_constant(f[xcol].to_numpy(float))
    r = sm.WLS(f["u"].to_numpy(float), X, weights=f["n"].to_numpy(float)).fit()
    slope = float(r.params[1])
    rng = np.random.default_rng(seed)
    null = []
    for _ in range(n_perm):
        xp = rng.permutation(f[xcol].to_numpy(float))
        null.append(sm.WLS(f["u"].to_numpy(float), sm.add_constant(xp), weights=f["n"].to_numpy(float)).fit().params[1])
    null = np.abs(np.array(null))
    return {"n_fields": int(len(f)), "slope": slope, "slope_se": float(r.bse[1]), "R2": float(r.rsquared),
            "perm_p_two_sided": float((1 + np.sum(null >= abs(slope))) / (1 + n_perm)), "n_perm": n_perm}


def mixed_icc(df: pd.DataFrame, fixed: list[str], vcs: list[str]) -> dict:
    """Latent-scale variance components from statsmodels BinomialBayesMixedGLM (variational Bayes)."""
    from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM
    from sklearn.preprocessing import StandardScaler
    d = df.copy()
    X = StandardScaler().fit_transform(S4._prep(d[fixed], np.ones(len(d), bool)))
    for i, c in enumerate(fixed):
        d[f"x{i}"] = X[:, i]
    d["concept_id"] = pd.factorize(d["concept"])[0]
    d["key_id"] = pd.factorize(d["key"])[0]
    fml = "R ~ " + " + ".join(f"x{i}" for i in range(len(fixed)))
    vc = {v: f"0 + C({v}_id)" for v in vcs}
    try:
        m = BinomialBayesMixedGLM.from_formula(fml, {k: vc[k] for k in vcs}, d.rename(columns={}), vcp_p=2.0)
        r = m.fit_vb()
    except (ValueError, np.linalg.LinAlgError) as e:
        logger.error(f"mixed GLM failed: {e}")
        return {"error": str(e)}
    sds = np.exp(r.vcp_mean)
    out = {f"tau2_{n}": float(s ** 2) for n, s in zip(m.vcp_names, sds)}
    tot = sum(out.values()) + math.pi ** 2 / 3
    for k in list(out):
        out[f"icc_latent_{k[5:]}"] = out[k] / tot
    return out


# ----------------------------------------------------------------------------- C1 rewiring
def _connected(E: dict, n: int = 26) -> bool:
    adj: dict[int, list[int]] = {i: [] for i in range(n)}
    for a, b in E:
        adj[a].append(b)
        adj[b].append(a)
    seen, stack = {0}, [0]
    while stack:
        for v in adj[stack.pop()]:
            if v not in seen:
                seen.add(v)
                stack.append(v)
    return len(seen) == n


def rewire(edges: dict, rng: np.random.Generator, nswap: int, shuffle_w: bool) -> dict:
    """Degree- and connectivity-preserving double-edge swaps with weights carried by their edges."""
    E = dict(edges)
    keys = list(E)
    done, tries = 0, 0
    while done < nswap and tries < nswap * 50:
        tries += 1
        i1, i2 = rng.choice(len(keys), 2, replace=False)
        (u, v), (x, y) = keys[i1], keys[i2]
        if rng.random() < 0.5:
            x, y = y, x
        if len({u, v, x, y}) < 4:
            continue
        e1, e2 = tuple(sorted((u, x))), tuple(sorted((v, y)))
        if e1 in E or e2 in E:
            continue
        w1, w2 = E.pop(keys[i1]), E.pop(keys[i2])
        E[e1], E[e2] = w1, w2
        if not _connected(E):
            del E[e1], E[e2]
            E[keys[i1]], E[keys[i2]] = w1, w2
            continue
        keys = list(E)
        done += 1
    if shuffle_w:
        ws = rng.permutation(list(E.values()))
        E = dict(zip(E.keys(), ws))
    return E


def eig_from_edges(E: dict) -> np.ndarray:
    import networkx as nx
    G = nx.Graph()
    G.add_nodes_from(range(26))
    for (a, b), w in E.items():
        G.add_edge(a, b, weight=w)
    e = nx.eigenvector_centrality_numpy(G, weight="weight")
    v = np.abs(np.array([e[i] for i in range(26)]))
    return v / v.max()

## Step 0: harmonise, then reproduce exp4 exactly
The code builds the backbone and the harmonised exp4 panel. It then re-runs exp4's own `field_level` routine and the
harmonised-panel `eval_specs`. Both must return **0.10254** for `gateway_j` over M0, and **0.10222** once field size is
also controlled. If either differs, the script stops. The same cell checks that the fast numpy `_prep` and AUC match the
originals.

*Demo change:* only the exp4 panel is built. The full run also loaded exp1, exp3, their union and the new-episodes panel
(`H.build_all()`). Here `pool`, the external row pool used by `P_pooled`, therefore holds only exp4's rows.

In [ ]:
t0 = time.time()
bb = Backbone()
d4, W4, c4 = load_exp4(bb)
d4["cl"] = d4["concept"]
pool = pd.concat([d4], ignore_index=True)[["key", "concept", "group", "R"]]  # full run: exp4 + exp3 + exp1 rows
DS = {"exp4": (d4, W4)}
logger.info(f"harmonisation checks exp4: {c4}")
for k, (d, _) in DS.items():
    logger.info(f"dataset {k}: rows={len(d)} concepts={d['concept'].nunique()} R-rate={d['R'].mean():.3f} "
                f"groups={d['group'].value_counts().to_dict()}")

# exact reproduction of exp4
fr4 = pd.DataFrame(data["field_outcomes"])  # notebook: was pd.read_csv(H.E4 / "field_outcomes.csv")
BF = ["log_n_W3", "growth_j", "share_W3"]
rep = {"gateway_j": S4.field_level(fr4, BF, BF + ["gateway_j"], n_boot=10)["delta_auc"],
       "size_controlled_gateway_j": S4.field_level(fr4, BF + ["log_field_size"],
                                                   BF + ["log_field_size", "gateway_j"], n_boot=10)["delta_auc"]}
rep_h = lib.eval_specs(d4, [("g", H.M0, H.M0 + ["gateway_j"]),
                            ("gs", H.M0 + ["log_field_size"], H.M0 + ["log_field_size", "gateway_j"])])
reproduction = {"reported": {"gateway_j": 0.10254, "size_controlled_gateway_j": 0.10222},
                "reproduced_from_exp4_file": rep,
                "reproduced_from_harmonised_panel": {"gateway_j": rep_h["g"]["delta"],
                                                     "size_controlled_gateway_j": rep_h["gs"]["delta"]}}
ok_rep = abs(rep["gateway_j"] - 0.10254) < 1e-4 and abs(rep["size_controlled_gateway_j"] - 0.10222) < 1e-4 and \
    abs(rep_h["g"]["delta"] - 0.10254) < 1e-4 and abs(rep_h["gs"]["delta"] - 0.10222) < 1e-4
reproduction["exact_to_1e-4"] = bool(ok_rep)
logger.info(f"reproduction: {reproduction}")
if not ok_rep:
    raise RuntimeError(f"exp4 reproduction failed: {reproduction}")

fast_eq = {}
for ds, (d, _) in DS.items():
    cols = spec_set(ds, False)[1]["M2"]
    tr = (d["group"] != "CS").to_numpy()
    a1, a2 = lib._ORIG_PREP(d[cols], tr), lib.fast_prep(d[cols], tr)
    oof = S4.logo_predict(d, cols, "R", "logit")
    fast_eq[ds] = {"prep_maxabs": float(np.abs(a1 - a2).max()),
                   "auc_diff": float(abs(lib._ORIG_AUC(d["R"].to_numpy(float), oof) -
                                         lib.fast_auc(d["R"].to_numpy(float), oof)))}
    assert fast_eq[ds]["prep_maxabs"] < 1e-12 and fast_eq[ds]["auc_diff"] < 1e-12, fast_eq
logger.info(f"fast-path equivalence: {fast_eq}")
reproduction["fast_path_equivalence"] = fast_eq

## Block A (with B1 and C3): replication under a concept-clustered refit bootstrap
For each spec, the code fits the baseline and baseline + candidate with LOGO and records the pooled delta-AUC. It then
repeats the fit on concept-resampled panels. The refit bootstrap re-estimates every model in every draw, so its CIs are
wider and more honest than iteration 1's fixed-prediction CIs. The specs are:
- **M0/M1/M2 + gateway**: the replication ladder.
- **M2+P + gateway** and **P_alone** (Block B1): does a leave-concept-out field propensity absorb the gateway signal?
- **rival:** entries (Block C3): each rival centrality is added to M2 in place of gateway.

If more than 5% of plain draws leave a test group with one class, the draws are **stratified by group**, as in the full run.

In [ ]:
A: dict = {}
pointres: dict = {}
for i, ds in enumerate(DS):
    d, _ = DS[ds]
    rivals = ds in ("exp4", "union", "new_eps")
    specs, fams = spec_set(ds, rivals)
    nb = N_BOOT_EXP4  # original: args.n_boot / n_boot_exp4 / n_boot_secondary by dataset

    def job(d=d, specs=specs, nb=nb, i=i):
        pt = lib.eval_specs(d, specs, 2.0, pool, keep_oof=True)
        bs = run_boot(d, specs, nb, pool, SEED + 100000 * i)
        share1 = np.mean([b["M2"]["nd"] > 0 for b in bs])
        strat = None
        if share1 > 0.05:
            strat = run_boot(d, specs, nb, pool, SEED + 100000 * i + 50000, stratified=True)
        return pt, bs, strat
    t_a = time.time()
    pt, bs, strat = job()
    logger.info(f"A_{ds}_{nb} done in {time.time() - t_a:.0f}s")
    pointres[ds] = pt
    res = summarise_boot(pt, strat if strat is not None else bs, specs)
    A[ds] = {"n_rows": int(len(d)), "n_concepts": int(d["concept"].nunique()), "prevalence": float(d["R"].mean()),
             "mean_episodes_per_concept": float(len(d) / d["concept"].nunique()),
             "rows_per_group": d["group"].value_counts().to_dict(), "feature_sets": fams, "n_boot": nb,
             "bootstrap_scheme": "stratified-by-group concept resampling (single-class draws > 5%)"
             if strat is not None else "concept-clustered refit bootstrap", "specs": res}
    if strat is not None:
        A[ds]["unstratified_M2_ci95"] = summarise_boot(pt, bs, specs)["M2"]["refit_boot"].get("ci95")
    # P sensitivity to shrinkage a
    sens = {}
    for a_ in (0.0, 5.0):
        r = lib.eval_specs(d, [s for s in specs if s[0] in ("M2+P", "P_alone")], a_, pool)
        sens[f"a={a_:g}"] = {k: {"delta": v["delta"], "auc_base": v["auc_base"], "auc_cand": v["auc_cand"]}
                             for k, v in r.items()}
    A[ds]["P_shrinkage_sensitivity"] = sens
    # descriptives
    A[ds]["spearman_gateway_with"] = {c: float(spearmanr(d["gateway_j"], d[c]).statistic)
                                      for c in ("log_field_size", "phi_home_j", "density_j")}
    m2 = specs[2][1]
    A[ds]["std_coef_gateway_in_M2"] = std_coef_boot(d, m2 + ["gateway_j"], "gateway_j", N_COEF_BOOT, SEED + i)
    # correlation gateway vs P across fields (full-data leave-concept-out P, a=2)
    Pw = lib.propensity(d["key"].to_numpy(), d["concept"].to_numpy(), d["R"].to_numpy(float),
                        d["group"].to_numpy(), "__none__", 2.0)
    A[ds]["spearman_gateway_vs_P_rows"] = float(spearmanr(d["gateway_j"], Pw).statistic)
    byf = pd.DataFrame({"k": d["key"], "g": d["gateway_j"], "P": Pw}).groupby("k").mean()
    A[ds]["spearman_gateway_vs_P_fields"] = float(spearmanr(byf["g"], byf["P"]).statistic) if len(byf) > 3 else None
    r = res["M2"]
    logger.info(f"A {ds}: M2 delta={r['delta']:.4f} ci95={r['refit_boot'].get('ci95')} "
                f"groups+={r['n_groups_positive']}/{r['n_groups_evaluable']}; +P delta={res['M2+P']['delta']:.4f} "
                f"P alone={res['P_alone']['delta']:.4f}")
# Note: the per-group "groups+" count can differ by one from the full run's 4/4: exp4's CS group has an exact AUC tie
# under M2 (full run: +1.1e-16, counted as positive), so its sign depends on the BLAS/sklearn build.
# random-effects pooling over exp4/exp1/exp3 needs all three files -> reported from the full run below
print(pd.DataFrame({k: {"delta": v["delta"], "ci95": v["refit_boot"].get("ci95"), "groups+": v["n_groups_positive"]}
                    for k, v in A["exp4"]["specs"].items()}).T.to_string())

## Block B2: is gateway a field trait? (field-intercept decomposition)
**Stage 1** fits M1 plus a dummy for each field and keeps each field's intercept `u_j`. **Stage 2** regresses `u_j` on
`gateway_j` with weighted least squares (weight = the field's row count) and tests the slope by permutation. The mixed GLMs
estimate how much of the field random-intercept variance `gateway_j` removes. A static field fixed-effect test of
`gateway_j` is unidentifiable: gateway is one number per field, so the field dummies absorb it completely. B3, the
time-varying gateway, needs exp3's raw scan files, so its full-run result is shown in the summary cell.

In [ ]:
B: dict = {"B1_propensity": {ds: {k: A[ds]["specs"][k] | {"per_group": None} for k in
                                  ("M2+P", "P_alone", "M2+Ppool", "Ppool_alone")} for ds in DS}}
B2: dict = {"static_FE_note": "gateway_j is one number per field -> perfectly collinear with field dummies; a static "
                              "field-FE test of gateway_j is unidentifiable by construction."}
for ds in ("exp4",):  # full run: ("exp4", "exp1", "exp3", "union", "new_eps")
    d, _ = DS[ds]
    M1 = spec_set(ds, False)[1]["M1"]
    fe = field_effects(d, M1)
    B2[ds] = {"stage2_gateway": wls_perm(fe, "gateway_j", N_WLS_PERM, SEED), "stage2_log_field_size":
              wls_perm(fe, "log_field_size", N_WLS_PERM, SEED + 1),
              "field_effects": fe.to_dict(orient="records")}
    B2["_fe_" + ds] = fe
    if ds in ("exp4", "union"):
        def icc_job(d=d, M1=M1):
            return {"field_and_concept_M1": mixed_icc(d, M1, ["key", "concept"]),
                    "field_and_concept_M1_plus_gateway": mixed_icc(d, M1 + ["gateway_j"], ["key", "concept"]),
                    "field_only_M1": mixed_icc(d, M1, ["key"]),
                    "field_only_M1_plus_gateway": mixed_icc(d, M1 + ["gateway_j"], ["key"])}
        ic = icc_job()
        try:
            t_a = ic["field_and_concept_M1"]["tau2_key"]
            t_b = ic["field_and_concept_M1_plus_gateway"]["tau2_key"]
            ic["share_tau2_field_removed_by_gateway"] = 1 - t_b / t_a if t_a > 0 else None
        except KeyError:
            ic["share_tau2_field_removed_by_gateway"] = None
        B2[ds]["icc"] = ic
    logger.info(f"B2 {ds}: slope={B2[ds]['stage2_gateway'].get('slope')} R2={B2[ds]['stage2_gateway'].get('R2')} "
                f"p={B2[ds]['stage2_gateway'].get('perm_p_two_sided')}  "
                f"share tau2 removed={B2[ds]['icc']['share_tau2_field_removed_by_gateway']}")
B["B2_field_intercepts"] = B2

## Block C: placebos and rival centralities
- **C1, rewiring null.** The backbone is rewired with degree- and connectivity-preserving double-edge swaps. The edge weights
  are either carried with their edges or shuffled. The eigenvector centrality is recomputed, and delta-AUC is re-estimated
  with the placebo gateway. The null is useful only if rewiring really moves the ranking: the median Spearman correlation
  between the real and rewired gateway must be at most 0.8.
- **C2, node-label permutation.** The 26 gateway values are shuffled across fields.
- **C3, rival centralities.** Each rival is added to M2. The resulting p-values get a Holm correction.

In [ ]:
C: dict = {}
G = bb.graph()
edges = {tuple(sorted((a, b))): w["weight"] for a, b, w in G.edges(data=True)}
for variant in ("carried", "weights_shuffled"):
    def c1_job(variant=variant):
        rng = np.random.default_rng(SEED + (1 if variant == "carried" else 2))
        vecs = []
        while len(vecs) < N_REWIRE:
            E = rewire(edges, rng, 10 * len(edges), variant == "weights_shuffled")
            try:
                vecs.append(eig_from_edges(E))
            except Exception as e:  # noqa: BLE001  eigen solver failure -> redraw with a new seed
                logger.warning(f"eig failed on rewired graph ({e}); redrawing")
        return vecs
    vecs = c1_job()
    rho = np.array([spearmanr(bb.gate, v).statistic for v in vecs])
    for ds in ("exp4",):  # full run: ("exp4", "union")
        d, W = DS[ds]
        fams = spec_set(ds, False)[1]
        nulls = [x for part in pool_map(lib.perm_worker, [(d, W, {"M0": fams["M0"], "M2": fams["M2"]}, list(c))
                                                          for c in np.array_split(np.array(vecs), N_WORKERS * 2)
                                                          if len(c)])
                 for x in part]
        for mdl in ("M0", "M2"):
            real = A[ds]["specs"][mdl]["delta"]
            nl = np.array([x[mdl] for x in nulls])
            C[f"C1_rewired_{variant}_{ds}_{mdl}"] = {
                "real": real, "null": nl, "null_median": float(np.median(nl)),
                "null_p95": float(np.percentile(nl, 95)), "real_percentile": float(np.mean(nl < real) * 100),
                "p_empirical": float((1 + np.sum(nl >= real)) / (1 + len(nl))), "n_draws": int(len(nl))}
    C[f"C1_discrimination_{variant}"] = {
        "median_spearman_real_vs_rewired": float(np.median(rho)), "p05": float(np.percentile(rho, 5)),
        "p95": float(np.percentile(rho, 95)),
        "flag": ("degree-preserving rewiring cannot separate eigenvector position from degree on a 26-node graph"
                 if np.median(rho) > 0.8 else "placebo discriminates (median rho <= 0.8)")}
    logger.info(f"C1 {variant}: median rho={np.median(rho):.3f}")
rng = np.random.default_rng(SEED + 3)
perms = [rng.permutation(bb.gate) for _ in range(N_PERM)]
for ds in ("exp4",):  # full run: ("exp4", "union", "new_eps")
    d, W = DS[ds]
    fams = spec_set(ds, False)[1]
    nulls = [x for part in pool_map(lib.perm_worker, [(d, W, {"M2": fams["M2"]}, list(c))
                                                      for c in np.array_split(np.array(perms), N_WORKERS * 3)
                                                      if len(c)])
             for x in part]
    nl = np.array([x["M2"] for x in nulls])
    real = A[ds]["specs"]["M2"]["delta"]
    C[f"C2_label_perm_{ds}_M2"] = {"real": real, "null": nl, "null_median": float(np.median(nl)),
                                   "null_p95": float(np.percentile(nl, 95)),
                                   "real_percentile": float(np.mean(nl < real) * 100),
                                   "p_empirical": float((1 + np.sum(nl >= real)) / (1 + len(nl))),
                                   "n_perm": int(len(nl))}
    logger.info(f"C2 {ds}: real={real:.4f} null p95={np.percentile(nl, 95):.4f} "
                f"percentile={C[f'C2_label_perm_{ds}_M2']['real_percentile']:.1f}")
# C3 rivals
C3 = {}
for ds in ("exp4",):  # full run: ("exp4", "union", "new_eps")
    sp = A[ds]["specs"]
    tab = {"gateway_j": {k: sp["M2"][k] for k in ("delta", "auc_base", "auc_cand")} | {"ci95": sp["M2"]["refit_boot"].get("ci95"),
                                                                                       "p_two_sided": sp["M2"]["refit_boot"].get("p_two_sided")}}
    for r in H.RIVALS:
        s = sp[f"rival:{r}"]
        tab[r] = {"delta": s["delta"], "auc_base": s["auc_base"], "auc_cand": s["auc_cand"],
                  "ci95": s["refit_boot"].get("ci95"), "p_two_sided": s["refit_boot"].get("p_two_sided")}
    hp = holm({k: v["p_two_sided"] for k, v in tab.items() if v["p_two_sided"] is not None})
    for k in tab:
        tab[k]["p_holm"] = hp.get(k)
    tab["_note"] = ("log_field_size row = M2 without log_field_size vs M2 (size is part of M2); "
                    f"gateway without size control: {sp['gateway_vs_M2_minus_size']['delta']:.4f}")
    C3[ds] = {"n_boot": A[ds]["n_boot"], "table": tab}
vecs = {k: v for k, v in bb.rivals.items()}
names = list(vecs)
C3["spearman_matrix_26_fields"] = {"names": names, "matrix": [[float(spearmanr(vecs[a], vecs[b]).statistic)
                                                               for b in names] for a in names]}
C["C3_rivals"] = C3
print(pd.DataFrame({k: v for k, v in C3["exp4"]["table"].items() if isinstance(v, dict)}).T[
    ["delta", "ci95", "p_two_sided", "p_holm"]].to_string())

## Block D: are the concept-level O1 gains a label-coverage artefact?
Iteration 1 reported that eight gateway-type concept features (G, G_all, G_deg, ...) improved the AUC for the concept-level
outcome O1. Block D re-scores each feature against three baselines: B5 alone, B5 plus `label_coverage_early`, and B5 plus
coverage plus a training-fold **O1 base rate** for the home field. A gain counts as an **artefact** when it shrinks by at
least 50% and its 90% CI covers 0.

*Notebook change:* O1 comes directly from `features`. The original first asserted that it equals `outcomes.csv`'s O1. The
reported iteration-1 deltas come from `data["exp4_screen_result"]`.

In [ ]:
def block_d():
    ft = pd.DataFrame(data["features"])  # notebook: was pd.read_csv(H.E4 / "features.csv") merged with outcomes.csv O1
    df = ft.copy()
    df = df.dropna(subset=["O1"]).reset_index(drop=True)
    df["cl"] = np.arange(len(df))
    B5c = ["log_count_W5", "growth_W5_B5", "offhome_share_W3", "entropy_W3", "reach_W3"]
    variants = ["G", "G_all", "G_deg", "G_btw", "G_phimin", "G_A", "REL_home", "DOM_Social"]
    specs = []
    for v in variants:
        miss = f"_miss_{v}"
        df[miss] = df[v].isna().astype(int)
        extra = [v] + ([miss] if df[miss].any() else [])
        for bn, base in (("B5", B5c), ("B5+cov", B5c + ["label_coverage_early"]),
                         ("B5+cov+O1base", B5c + ["label_coverage_early", "O1_base"])):
            specs.append((f"{v}|{bn}", base, base + extra))
    pt = lib.concept_specs_eval(df, specs)
    seeds = list(range(SEED + 9000, SEED + 9000 + N_BOOT_D))
    bs = [x for part in pool_map(lib.concept_boot_worker, [(df, specs, c) for c in chunks(seeds, N_WORKERS * 2)])
          for x in part]
    return df, specs, pt, bs
D: dict = {}
dfD, specsD, ptD, bsD = block_d()
s4 = data["exp4_screen_result"]  # notebook: was json.loads((H.E4 / "screen_result.json").read_text())
for nm, _, _ in specsD:
    v, bn = nm.split("|")
    arr = np.array([b[nm] for b in bsD])
    D.setdefault(v, {})[bn] = {**ptD[nm], **{k: x for k, x in ci(arr).items()},
                               "n_groups_positive": int(sum(1 for x in ptD[nm]["per_group"].values()
                                                            if np.isfinite(x) and x > 0))}
for v in D:
    rep_v = (s4["secondary_screens"].get(v, {}).get("O1", {}) or {}).get("delta") if v != "G" else \
        s4["delta_auc_O1"]["delta"]
    D[v]["reported_iter1_delta"] = rep_v
    D[v]["reproduces_iter1"] = bool(rep_v is not None and abs(rep_v - D[v]["B5"]["delta"]) < 1e-6)
    d0, d2 = D[v]["B5"]["delta"], D[v]["B5+cov+O1base"]["delta"]
    d1_ = D[v]["B5+cov"]["delta"]
    art = lambda dd, c90: bool((d0 > 0) and (dd <= 0.5 * d0) and c90[0] <= 0 <= c90[1])  # noqa: E731
    D[v]["artefact_cov"] = art(d1_, D[v]["B5+cov"].get("ci90", [0, 0]))
    D[v]["artefact_cov_O1base"] = art(d2, D[v]["B5+cov+O1base"].get("ci90", [0, 0]))
    x = dfD[v].to_numpy(float)
    cov = dfD["label_coverage_early"].to_numpy(float)
    y = dfD["O1"].to_numpy(float)
    ok = np.isfinite(x) & np.isfinite(cov)
    D[v]["spearman_with_label_coverage"] = float(spearmanr(x[ok], cov[ok]).statistic)
    rx, rc, ry = [pd.Series(a[ok]).rank().to_numpy() for a in (x, cov, y)]
    ex_ = rx - np.polyval(np.polyfit(rc, rx, 1), rc)
    ey_ = ry - np.polyval(np.polyfit(rc, ry, 1), rc)
    D[v]["partial_spearman_with_O1_given_coverage"] = float(np.corrcoef(ex_, ey_)[0, 1])
    D[v]["spearman_with_O1"] = float(spearmanr(x[ok], y[ok]).statistic)
D["_n_concepts"] = int(len(dfD))
D["_n_boot"] = N_BOOT_D
logger.info("D: " + "; ".join(f"{v}: {D[v]['B5']['delta']:.3f}->{D[v]['B5+cov+O1base']['delta']:.3f}"
                              for v in D if not v.startswith("_")))
print(pd.DataFrame({v: {"iter1": x["reported_iter1_delta"], "reproduces": x["reproduces_iter1"],
                        "B5": x["B5"]["delta"], "B5+cov": x["B5+cov"]["delta"],
                        "B5+cov+O1base": x["B5+cov+O1base"]["delta"], "artefact": x["artefact_cov_O1base"]}
                    for v, x in D.items() if not v.startswith("_")}).T.to_string())

## Block E: power. How many episodes would detect a true delta-AUC of 0.05?
- **Analytic MDE.** The refit-bootstrap SE is scaled by a design effect `1 + (m - 1) rho_c`, where m is the number of
  episodes per concept and `rho_c` is the concept ICC.
- **Simulation.** Clustered episodes are drawn from the fitted M2 + gateway logit with a concept random intercept. The
  gateway coefficient is set either to 0 (null) or to the value that gives a true delta-AUC of 0.05 (alternative).
  A second run adds a **field random intercept**. With only 26 fields, that term sets a floor under the MDE that more
  episodes cannot remove.

*Demo change:* the full run drives Block E with the **union** panel (SE, ICC and `tau_f` all come from union). Here the
same
code runs on the exp4 panel (`du = DS["exp4"][0]`), so the absolute MDEs differ. The mechanics and the field-RE floor are
the same.

In [ ]:
E: dict = {}
du = DS["exp4"][0]                           # full run: DS["union"][0]
m2u = spec_set("exp4", False)[1]["M2"]       # full run: spec_set("union", False)[1]["M2"]
se_boot = A["exp4"]["specs"]["M2"]["refit_boot"]["sd"]
N0, nc0 = len(du), du["concept"].nunique()
m0 = N0 / nc0
# ICC of R within concept: latent-scale concept random intercept (M2 fixed) and ANOVA on Pearson residuals
icc_c = mixed_icc(du, m2u, ["concept"])
rho_latent = icc_c.get("icc_latent_concept", math.nan)
oofm = pointres["exp4"]["M2"]["oof_cand"]
pr = (du["R"].to_numpy() - oofm) / np.sqrt(np.clip(oofm * (1 - oofm), 1e-6, None))
grp = pd.Series(pr).groupby(du["concept"].to_numpy())
k = grp.size()
msb = (k * (grp.mean() - pr.mean()) ** 2).sum() / (len(k) - 1)
msw = ((pr - grp.transform("mean").to_numpy()) ** 2).sum() / (len(pr) - len(k))
n_bar = (len(pr) - (k ** 2).sum() / len(pr)) / (len(k) - 1)
rho_anova = float(max(0.0, (msb - msw) / (msb + (n_bar - 1) * msw)))
rho_c = rho_latent if np.isfinite(rho_latent) else rho_anova
DE = lambda m: 1 + (m - 1) * rho_c  # noqa: E731
se_fn = lambda N, m: se_boot * math.sqrt((N0 / DE(m0)) / (N / DE(m)))  # noqa: E731
mde_fn = lambda N, m: (1.96 + 0.84) * se_fn(N, m)  # noqa: E731
shrunk = A["exp4"]["specs"]["M2"]["refit_boot"]["ci90"][0]
E["inputs"] = {"SE_boot_M2": se_boot, "N0_rows": N0, "n_concepts": nc0, "m0": m0,
               "rho_c_latent": rho_latent, "rho_c_anova_pearson": rho_anova, "rho_c_used": rho_c,
               "rho_c_source": "latent" if np.isfinite(rho_latent) else "anova",
               "shrunken_effect_lower90": shrunk, "H1_bar": 0.05}
E["analytic"] = [{"N": N, "m": m, "DE": DE(m), "SE": se_fn(N, m), "MDE_80": mde_fn(N, m),
                  "power_at_0.05": float(norm.cdf(0.05 / se_fn(N, m) - 1.96)),
                  "power_at_shrunken": float(norm.cdf(max(shrunk, 0) / se_fn(N, m) - 1.96))}
                 for N in (1000, 2000, 4000) for m in (5, 10)]
E["_mde_fn"] = mde_fn
logger.info(f"E inputs: {E['inputs']}")
print(pd.DataFrame(E["analytic"]).to_string())

In [ ]:
def sim_job():
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    cols = m2u + ["gateway_j"]
    X = S4._prep(du[cols], np.ones(len(du), bool))
    sc = StandardScaler().fit(X)
    mdl = LogisticRegression(C=1.0, max_iter=2000).fit(sc.transform(X), du["R"].to_numpy(int))
    beta = mdl.coef_[0] / sc.scale_
    b0 = mdl.intercept_[0] - np.sum(mdl.coef_[0] * sc.mean_ / sc.scale_)
    sig = math.sqrt(rho_c * (math.pi ** 2 / 3) / (1 - rho_c)) if 0 < rho_c < 1 else 0.0
    gi = len(cols) - 1
    sg_, mg_ = float(X[:, gi].std()), float(X[:, gi].mean())

    def with_bstd(bstd):
        bb_ = beta.copy()
        bb_[gi] = bstd / sg_
        return b0 + (beta[gi] - bb_[gi]) * mg_, bb_

    keyc = pd.factorize(du["key"])[0]

    def run(bstd, N, m, n, base_seed, tau_f=0.0):
        b0_, be_ = with_bstd(bstd)
        seeds = list(range(base_seed, base_seed + n))
        return np.array([x for part in pool_map(lib.sim_worker, [(X, b0_, be_, gi, sig, N, m, list(c), keyc, tau_f)
                                                                 for c in chunks(seeds, N_WORKERS * 2)])
                         for x in part])
    # calibrate the standardised gateway coefficient that yields a true grouped-CV delta-AUC of 0.05
    grid = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0]
    pil = {b: float(run(b, 2000, 5, N_PILOT, SEED + 31 + int(b * 100) * 1000).mean()) for b in grid}
    xs, ys = np.array(grid), np.array([pil[b] for b in grid])
    order = np.argsort(ys)
    b_star = float(np.interp(0.05, ys[order], xs[order]))
    logger.info(f"E calibration: {pil} -> b_std*={b_star:.3f}")
    out = []
    for N in SIM_NS:
        for m in SIM_MS:
            nul = run(0.0, N, m, N_SIM, SEED + N * 10 + m)
            alt = run(b_star, N, m, N_SIM, SEED + N * 10 + m + 500000)
            crit = float(np.percentile(nul, 95))
            out.append({"N": N, "m": m, "n_sims_null": int(len(nul)), "n_sims_alt": int(len(alt)),
                        "SD_null": float(nul.std(ddof=1)), "crit95_null": crit,
                        "mean_delta_alt": float(alt.mean()), "SD_alt": float(alt.std(ddof=1)),
                        "power_at_0.05_sim": float(np.mean(alt > crit)),
                        "mde_sim": float(crit + 0.84 * alt.std(ddof=1)),
                        "power_at_0.05_normal_approx": float(norm.cdf((alt.mean() - crit) / alt.std(ddof=1)))})
            logger.info(f"E sim N={N} m={m}: crit={crit:.4f} alt mean={alt.mean():.4f} sd={alt.std(ddof=1):.4f} "
                        f"power={np.mean(alt > crit):.3f}")
    # sensitivity: add a field random intercept with the field variance (field-only M1 model, B2)
    tau_f = math.sqrt(max(tau2_field_union, 0.0))
    pil_f = {b: float(run(b, 2000, 5, N_PILOT, SEED + 77 + int(b * 100) * 1000, tau_f).mean()) for b in grid}
    ysf = np.array([pil_f[b] for b in grid])
    of = np.argsort(ysf)
    b_star_f = float(np.interp(0.05, ysf[of], xs[of]))
    out_f = []
    for N in SIM_NS:
        for m in (5,):
            nul = run(0.0, N, m, N_SIM, SEED + N * 10 + m + 7, tau_f)
            alt = run(b_star_f, N, m, N_SIM, SEED + N * 10 + m + 500007, tau_f)
            crit = float(np.percentile(nul, 95))
            out_f.append({"N": N, "m": m, "SD_null": float(nul.std(ddof=1)), "crit95_null": crit,
                          "mean_delta_alt": float(alt.mean()), "SD_alt": float(alt.std(ddof=1)),
                          "power_at_0.05_sim": float(np.mean(alt > crit)),
                          "mde_sim": float(crit + 0.84 * alt.std(ddof=1))})
            logger.info(f"E sim fieldRE N={N} m={m}: crit={crit:.4f} alt mean={alt.mean():.4f} "
                        f"sd={alt.std(ddof=1):.4f} power={np.mean(alt > crit):.3f}")
    return {"sigma_concept": sig, "fitted_std_coef_gateway": float(mdl.coef_[0][gi]),
            "field_RE_sensitivity": {"tau_field": tau_f, "tau2_source": "field-only M1 random intercept (B2)",
                                     "calibration_pilot": pil_f, "b_std_for_delta_0.05": b_star_f,
                                     "cells": out_f},
            "calibration_pilot_mean_delta_by_bstd": pil, "b_std_for_delta_0.05": b_star, "cells": out}
tau2_field_union = B2["exp4"]["icc"]["field_only_M1"].get("tau2_key", 0.0)  # full run: B2["union"]
t_e = time.time()
sim = sim_job()
logger.info(f"E simulation done in {time.time() - t_e:.0f}s")
E["simulation"] = sim["cells"]
E["simulation_meta"] = {k: v for k, v in sim.items() if k != "cells"}
print(pd.DataFrame(E["simulation"]).to_string())
print(pd.DataFrame(E["simulation_meta"]["field_RE_sensitivity"]["cells"]).to_string())

## Verdict (pre-registered ladder)
The ladder is judged on the **union** and **new-episodes** panels, which the demo does not rebuild. The cell applies the
original ladder code to the full run's recorded conditions. It also prints this demo's exp4 numbers beside the full run's
exp4 numbers for comparison.

In [ ]:
REF = data["reference_full_run"]
cond = {k: bool(v) for k, v in REF["summary"]["conditions"].items()}
if not cond["new_eps_delta_gt_0"]:
    verdict = "FAILS"
elif (cond["P_alone_carries_gain_union"] and cond["gateway_adds_le_0.01_given_P_union"]) or cond["inside_C2_null_union"]:
    verdict = "FIELD-TRAIT"
elif all(cond[k] for k in ("union_delta_gt_0_ci95_gt_0", "new_eps_delta_gt_0_ci95_gt_0",
                            "union_ge3of4_groups_positive", "survives_P_within_union_ci95_gt_0",
                            "above_C2_p95_union")):
    verdict = "REPLICATES"
elif REF["metrics_agg"]["A_union_M2_delta_auc"] > 0 and REF["metrics_agg"]["A_new_eps_M2_delta_auc"] > 0:
    verdict = "ATTENUATES"
else:
    verdict = "FAILS"
logger.info(f"VERDICT (full-run conditions): {verdict}  {cond}")
assert verdict == REF["summary"]["verdict"]
print("runtime so far: %.0fs" % (time.time() - t0))

## Results summary and visualisation

In [ ]:
MA = REF["metrics_agg"]
# 1) demo exp4 vs full-run exp4 (point estimates are deterministic and must agree; CIs depend on the draw count)
rows = []
for sp_ in ("M0", "M1", "M2", "M2+P", "P_alone"):
    s = A["exp4"]["specs"][sp_]
    tag = f"exp4_{sp_.replace('+', '_plus_')}"
    rows.append({"spec": sp_, "demo delta": s["delta"], "demo 95% CI": np.round(s["refit_boot"].get("ci95", [np.nan] * 2), 3),
                 "full delta": MA.get(f"A_{tag}_delta_auc"),
                 "full 95% CI": np.round([MA.get(f"A_{tag}_ci95_lo", np.nan), MA.get(f"A_{tag}_ci95_hi", np.nan)], 3)})
print(f"exp4 panel: delta-AUC of gateway_j (demo: {N_BOOT_EXP4} refit draws; full run: 2000)")
print(pd.DataFrame(rows).to_string(index=False))

# 2) the replication across panels (full run) - the part that decides the verdict
print("\nFull run, delta-AUC of gateway_j over M2 (refit concept bootstrap 95% CI):")
head = REF["summary"]["headline"]
print(pd.DataFrame({ds: {"delta": v["delta"], "ci95_lo": v["ci95"][0], "ci95_hi": v["ci95"][1],
                         "AUC M2": v["auc_base"], "groups+": v["n_groups_positive"]}
                    for ds, v in head.items()}).T.round(4).to_string())
print(f"\nB2 exp4 (demo): stage-2 slope={B2['exp4']['stage2_gateway']['slope']:.2f}, "
      f"R2={B2['exp4']['stage2_gateway']['R2']:.2f}, perm p={B2['exp4']['stage2_gateway']['perm_p_two_sided']:.3f}  |  "
      f"full-run union: R2={MA['B2_union_stage2_gateway_R2']:.3f}, p={MA['B2_union_stage2_gateway_perm_p_two_sided']:.2f}")
print(f"B3 (full run): within/between SD ratio={MA['B3_sd_ratio_within_between']:.3f} (< 0.10 gate -> NOT IDENTIFIABLE); "
      f"slice-vs-exp4 Spearman={MA['B3_validation_spearman']:.2f}")
print(f"C2 exp4 (demo, {N_PERM} perms): real at {C['C2_label_perm_exp4_M2']['real_percentile']:.1f}th pct  |  "
      f"full-run union: {MA['C2_label_perm_union_M2_real_percentile']:.1f}th pct")
print(f"VERDICT: {verdict}")

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(15, 8.5))
# (a) forest: full-run M2 deltas per panel + the demo's exp4 estimate
ax = axs[0, 0]
labs = list(head)
for i, ds in enumerate(labs):
    y = len(labs) - i
    ax.plot(head[ds]["ci95"], [y, y], color="k", lw=1.2)
    ax.plot(head[ds]["delta"], y, "o", color="k", ms=4)
dm = A["exp4"]["specs"]["M2"]
yd = len(labs) - labs.index("exp4") - 0.3
ax.plot(dm["refit_boot"].get("ci95", [np.nan] * 2), [yd, yd], color="tab:orange", lw=1.2)
ax.plot(dm["delta"], yd, "s", color="tab:orange", ms=4, label=f"demo exp4 ({N_BOOT_EXP4} draws)")
ax.set_yticks(range(len(labs), 0, -1), labs)
ax.axvline(0, color="r", ls="--", lw=0.8)
ax.set_xlabel("delta-AUC gateway over M2 (95% refit CI)")
ax.set_title("(a) Replication across panels (black: full run)")
ax.legend(fontsize=7, loc="lower right")

# (b) exp4 ladder: M0 -> M1 -> M2 -> M2+P
ax = axs[0, 1]
lad = ["M0", "M1", "M2", "M2+P"]
dv = [A["exp4"]["specs"][s]["delta"] for s in lad]
cis = np.array([A["exp4"]["specs"][s]["refit_boot"].get("ci95", [np.nan] * 2) for s in lad])
ax.bar(lad, dv, color=["tab:blue", "tab:blue", "tab:blue", "tab:gray"])
ax.errorbar(lad, dv, yerr=[np.array(dv) - cis[:, 0], cis[:, 1] - np.array(dv)], fmt="none", ecolor="k", capsize=3)
ax.axhline(0, color="k", lw=0.6)
ax.set_ylabel("delta-AUC (exp4, demo)")
ax.set_title("(b) exp4: the lead shrinks as baselines grow")

# (c) C2 / C1 null distributions on exp4
ax = axs[0, 2]
for key, col in (("C2_label_perm_exp4_M2", "0.6"), ("C1_rewired_carried_exp4_M0", "tab:green")):
    v = C[key]
    ax.hist(v["null"], bins=20, color=col, alpha=0.6, label=f"{key.split('_exp4')[0]} null ({len(v['null'])})")
    ax.axvline(v["real"], color=col, lw=2, ls="--")
ax.set_xlabel("delta-AUC")
ax.set_title("(c) placebo nulls vs real (dashed), exp4")
ax.legend(fontsize=7)

# (d) stage-2 field intercepts vs gateway
ax = axs[1, 0]
fe = B2["_fe_exp4"]
f = fe[fe["n"] >= 3]
ax.scatter(f["gateway_j"], f["u"], s=8 + 6 * f["n"], alpha=0.7)
for _, r in f.iterrows():
    ax.annotate(str(r["key"])[:14], (r["gateway_j"], r["u"]), fontsize=6)
ax.set_xlabel("gateway_j (eigenvector, 1998-2002 backbone)")
ax.set_ylabel("field intercept u_j (stage 1)")
ax.set_title(f"(d) B2 exp4: R2={B2['exp4']['stage2_gateway']['R2']:.2f}, "
             f"p={B2['exp4']['stage2_gateway']['perm_p_two_sided']:.2f}")

# (e) Block D: O1 gain before/after coverage + O1-base controls
ax = axs[1, 1]
vs = [v for v in D if not v.startswith("_")]
xx = np.arange(len(vs))
ax.bar(xx - 0.2, [D[v]["B5"]["delta"] for v in vs], 0.4, label="over B5 (iteration 1)")
ax.bar(xx + 0.2, [D[v]["B5+cov+O1base"]["delta"] for v in vs], 0.4, label="over B5 + coverage + O1 base")
ax.set_xticks(xx, vs, rotation=45, fontsize=7)
ax.axhline(0, color="k", lw=0.6)
ax.set_ylabel("delta-AUC for O1")
ax.set_title("(e) Block D: O1 gains are coverage artefacts")
ax.legend(fontsize=7)

# (f) MDE vs N
ax = axs[1, 2]
Ns = np.linspace(300, 6000, 60)
for m in (5, 10):
    ax.plot(Ns, [E["_mde_fn"](n, m) for n in Ns], label=f"analytic, m={m}")
    sims = [(c["N"], c["mde_sim"]) for c in E.get("simulation", []) if c["m"] == m and c.get("mde_sim")]
    if sims:
        ax.plot(*zip(*sims), "o", ms=4, label=f"simulated, m={m}")
fre = E["simulation_meta"]["field_RE_sensitivity"]["cells"]
ax.plot([c["N"] for c in fre], [c["mde_sim"] for c in fre], "s--", ms=4, color="k", label="simulated + field RE, m=5")
ax.axhline(0.05, color="r", ls="--", lw=0.8, label="H1 bar 0.05")
ax.set_xlabel("episodes N")
ax.set_ylabel("MDE delta-AUC (80% power)")
ax.set_title("(f) Block E power (exp4-driven demo)")
ax.legend(fontsize=6)
fig.suptitle(f"Gateway-field retention stress test - verdict: {verdict}", fontsize=12)
fig.tight_layout()
plt.show()
print("total runtime: %.0fs" % (time.time() - t0))